## Project Description: Next Word Prediction Using LSTM
#### Project Overview:

This project aims to develop a deep learning model for predicting the next word in a given sequence of words. The model is built using Long Short-Term Memory (LSTM) networks, which are well-suited for sequence prediction tasks. The project includes the following steps:

1- Data Collection: We use the text of Shakespeare's "Hamlet" as our dataset. This rich, complex text provides a good challenge for our model.

2- Data Preprocessing: The text data is tokenized, converted into sequences, and padded to ensure uniform input lengths. The sequences are then split into training and testing sets.

3- Model Building: An LSTM model is constructed with an embedding layer, two LSTM layers, and a dense output layer with a softmax activation function to predict the probability of the next word.

4- Model Training: The model is trained using the prepared sequences, with early stopping implemented to prevent overfitting. Early stopping monitors the validation loss and stops training when the loss stops improving.

5- Model Evaluation: The model is evaluated using a set of example sentences to test its ability to predict the next word accurately.

6- Deployment: A Streamlit web application is developed to allow users to input a sequence of words and get the predicted next word in real-time.

In [2]:
#data collection 

import nltk
nltk.download('gutenberg')
from nltk.corpus import gutenberg
import pandas as pd

#lode the data
data=gutenberg.raw('shakespeare-hamlet.txt')
#save the data to a csv file
with open('hamlet.txt', 'w') as f:
    f.write(data)

[nltk_data] Downloading package gutenberg to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!


In [3]:
#data preprocessing
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split


In [4]:
#lode the data set
with open('hamlet.txt', 'r') as f:
    text=f.read().lower()

#tokenize the data
tokenizer=Tokenizer()
tokenizer.fit_on_texts([text])
total_words=len(tokenizer.word_index)+1
total_words


4818

In [5]:
#tokenizer.word_index

In [6]:
#create input sequences
input_sequences=[]
for line in text.split('\n'):
    token_list=tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence=token_list[:i+1]
        input_sequences.append(n_gram_sequence)

In [7]:
input_sequences[:10]

[[1, 687],
 [1, 687, 4],
 [1, 687, 4, 45],
 [1, 687, 4, 45, 41],
 [1, 687, 4, 45, 41, 1886],
 [1, 687, 4, 45, 41, 1886, 1887],
 [1, 687, 4, 45, 41, 1886, 1887, 1888],
 [1180, 1889],
 [1180, 1889, 1890],
 [1180, 1889, 1890, 1891]]

In [8]:
# pad the sequences
max_sequence_len=max([len(x) for x in input_sequences])
input_sequences=pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre')
input_sequences[:10]

array([[   0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    1,  687],
       [   0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           1,  687,    4],
       [   0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    1,
         687,    4,   45],
       [   0,    0,    0,    0,    0,    0,    0,    0,    0,    1,  687,
           4,   45,   41],
       [   0,    0,    0,    0,    0,    0,    0,    0,    1,  687,    4,
          45,   41, 1886],
       [   0,    0,    0,    0,    0,    0,    0,    1,  687,    4,   45,
          41, 1886, 1887],
       [   0,    0,    0,    0,    0,    0,    1,  687,    4,   45,   41,
        1886, 1887, 1888],
       [   0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0, 1180, 1889],
       [   0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
        1180, 1889, 1890],
       [   0,    0,    0,    0,    0,    0,    0,    0,    0,    0, 1180,
        1889, 189

In [9]:
# create predictors and labels
import tensorflow as tf
x,y=input_sequences[:,:-1], input_sequences[:,-1]
y=tf.keras.utils.to_categorical(y, num_classes=total_words) 



In [10]:
y

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(25732, 4818))

In [11]:
x

array([[   0,    0,    0, ...,    0,    0,    1],
       [   0,    0,    0, ...,    0,    1,  687],
       [   0,    0,    0, ...,    1,  687,    4],
       ...,
       [   0,    0,    0, ...,  687,    4,   45],
       [   0,    0,    0, ...,    4,   45, 1047],
       [   0,    0,    0, ...,   45, 1047,    4]],
      shape=(25732, 13), dtype=int32)

In [12]:
#split the data into training and testing sets
x_train, x_test, y_train, y_test=train_test_split(x,y, test_size=0.2, random_state=42)


In [13]:
#train the lstm rnn model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense,Dropout
model=Sequential()


In [14]:
model.add(Embedding(total_words, 100, input_length=max_sequence_len))
model.add(LSTM(150,return_sequences=True))
model.add(Dropout(0.2))
model.add(LSTM(100))
model.add(Dense(total_words, activation='softmax'))
model.compile(loss='categorical_crossentropy', optimizer='adam',metrics=['accuracy'])
model.build(input_shape=(None, max_sequence_len-1))
model.summary()

c:\Users\HP\anaconda3\envs\py312\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 13, 100)        │       481,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 13, 150)        │       150,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 13, 150)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 100)            │       100,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 4818)           │       486,618 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,219,418 (4.65 MB)

 Trainable params: 1,219,418 (4.65 MB)

 Non-trainable params: 0 (0.00 B)

In [15]:
history=model.fit(x_train, y_train, epochs=50, validation_data=(x_test, y_test),verbose=1)


Epoch 1/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 14s 17ms/step - accuracy: 0.0330 - loss: 6.9085 - val_accuracy: 0.0336 - val_loss: 6.6895
Epoch 2/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 10s 15ms/step - accuracy: 0.0392 - loss: 6.4725 - val_accuracy: 0.0408 - val_loss: 6.7472
Epoch 3/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 10s 15ms/step - accuracy: 0.0462 - loss: 6.3355 - val_accuracy: 0.0492 - val_loss: 6.7892
Epoch 4/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 10s 15ms/step - accuracy: 0.0515 - loss: 6.1910 - val_accuracy: 0.0499 - val_loss: 6.8068
Epoch 5/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 10s 16ms/step - accuracy: 0.0547 - loss: 6.0483 - val_accuracy: 0.0573 - val_loss: 6.8434
Epoch 6/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 10s 16ms/step - accuracy: 0.0611 - loss: 5.8981 - val_accuracy: 0.0598 - val_loss: 6.9177
Epoch 7/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 10s 16ms/step - accuracy: 0.0692 - loss: 5.7528 - val_accuracy: 0.0647 - val_loss: 6.9558
Epoch 8/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 10s 15ms/step - accuracy: 0.0800 - loss: 5.6115 - 

In [19]:
#function  to predict the next word in a sequence
def predict_next_word(model, tokenizer, text, max_sequence_len):
    token_list=tokenizer.texts_to_sequences([text])[0]
    if len(token_list)>=max_sequence_len:
        token_list=token_list[-(max_sequence_len-1):]
    token_list=pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')
    predicted=model.predict(token_list, verbose=0)
    predict_word_index=np.argmax(predicted,axis=1)
    for word, index in tokenizer.word_index.items():
        if index==predict_word_index:
            return word
    return None


In [22]:
input_text="To be or not to be"
print(f'input text: {input_text}')
max_sequence_len=model.input_shape[1]+1
next_word=predict_next_word(model, tokenizer, input_text, max_sequence_len)
print(f'predicted next word: {next_word}')

input text: To be or not to be
predicted next word: buried


In [23]:
#save model 
model.save('lstm_model.h5')
#save the tokenizer
import pickle
with open('tokenizer.pickle', 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)
    

In [26]:
input_text="mar. Horatio saies ,'tis but our"
print(f'input text: {input_text}')
max_sequence_len=model.input_shape[1]+1
next_word=predict_next_word(model, tokenizer, input_text, max_sequence_len)
print(f'predicted next word: {next_word}')


input text: mar. Horatio saies ,'tis but our
predicted next word: fantasie
